# Qwen Image Edit — synthetic person insertion (demo)

A thin walkthrough of the generation pipeline. It runs on either:

- a **single large-memory GPU** (H100/B200, 80GB), where the whole
  Qwen-Image-Edit-2509 pipeline fits in bf16 — the default, and
- a **multi-GPU instance** that can't fit the model on one device
  (e.g. 4×A10G g5.12xlarge), by flipping `MODEL_PARALLEL = True` at the load step.

The single- vs multi-GPU split is the only difference, and it's hidden behind the
`model_parallel` flag on `load_pipeline` (see `model.py`).

It reuses the shared modules (`config`, `prompts`, `recognition`, `s3_io`, `model`)
so this notebook stays a demo; the unattended batch job lives in `generate_synthetic.py`
(pass `--model-parallel` there for the multi-GPU case).

**Prerequisites**
- From the repository root, install dependencies with `uv sync --frozen --project scripts`.
- Register the environment as described in the README, then select the
  `Python (synthetic generation)` kernel.
- Replace the demo `SDA_S3_BUCKET` value in the first code cell with your bucket.
- The OpenImages subset must already be in S3 (see `data_prep/`).

In [ ]:
import os

os.environ["SDA_S3_BUCKET"] = "amzn-s3-demo-bucket"
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")

In [ ]:
import boto3

from config import require_bucket
from model import load_pipeline, edit_image
from prompts import build_prompt
from recognition import extract_person_boxes, dedupe_person_boxes
from s3_io import ImageResizer, read_image_from_s3

S3_BUCKET = require_bucket()
SOURCE_IMAGES = "datasets/openimages_subset/images/train_original/"

s3 = boto3.client("s3")
rekognition = boto3.client("rekognition")

## Pick a source image

List a few train images (no people) from the prepared subset in S3.

In [ ]:
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=SOURCE_IMAGES, MaxKeys=5)
image_ids = [obj["Key"].split("/")[-1].replace(".jpg", "") for obj in response.get("Contents", [])]
print(image_ids)

image_id = image_ids[0]
image = read_image_from_s3(s3, S3_BUCKET, image_id, SOURCE_IMAGES)
image

## Load the pipeline

Loads Qwen-Image-Edit-2509 in bf16. On a single large-memory GPU (H100/B200, 80GB)
leave `MODEL_PARALLEL = False` to load the whole pipeline onto `cuda`. On a
multi-GPU instance that can't fit the model on one device (e.g. 4×A10G g5.12xlarge),
set `MODEL_PARALLEL = True` and `load_pipeline` hand-shards the transformer across
the visible GPUs. The sharding details are hidden in `model.py`.

> **Note:** the model-parallel path is *slower per image* than a single-GPU load —
> activations are copied between GPUs at each layer boundary (inter-device
> communication). Use it only when the model doesn't fit on one GPU, not for speed.

In [ ]:
# Set True on a multi-GPU instance (e.g. 4xA10G g5.12xlarge), False on one big GPU.
# Note: True is slower per image (inter-device communication) — use only if the
# model doesn't fit on a single GPU.
MODEL_PARALLEL = False

pipeline = load_pipeline(model_parallel=MODEL_PARALLEL)
resizer = ImageResizer(min_dim=512)

## Edit: insert a person

Build a prompt (hazardous placement, no scene augmentation here) and run the edit
at 40 inference steps.

In [ ]:
prompt = build_prompt(gender="male", placement="hazardous", scene_augmentation=False)

output_image = edit_image(pipeline, resizer.resize(image), prompt, seed=2025)
output_image = resizer.restore(output_image)
output_image

## Pseudo-label with Amazon Rekognition

Detect the inserted person, dedupe boxes with NMS, and draw them for a sanity check.

In [ ]:
from io import BytesIO
from PIL import ImageDraw

# Force RGB so a non-RGB source (grayscale/CMYK/RGBA) still encodes cleanly as
# JPEG -- matches generate_synthetic.py's batch path.
buffer = BytesIO()
output_image.convert("RGB").save(buffer, format="JPEG")

response = rekognition.detect_labels(
    Image={"Bytes": buffer.getvalue()},
    MaxLabels=10,
    MinConfidence=80,
    Features=["GENERAL_LABELS"],
)
person_boxes = dedupe_person_boxes(extract_person_boxes(response["Labels"]))
print(f"{len(person_boxes)} person box(es) detected")

annotated = output_image.copy()
draw = ImageDraw.Draw(annotated)
w, h = annotated.size
for p in person_boxes:
    b = p["bbox"]
    x1, y1 = int(b["Left"] * w), int(b["Top"] * h)
    x2, y2 = int((b["Left"] + b["Width"]) * w), int((b["Top"] + b["Height"]) * h)
    draw.rectangle([x1, y1, x2, y2], outline="lime", width=3)
    draw.text((x1, max(0, y1 - 15)), f"{p['label']} {p['confidence']:.0f}%", fill="lime")
annotated

## Next steps

To generate a full synthetic dataset unattended (all four ablation conditions,
resume cache, S3 upload, YOLO label merge):

From the repository root, run the batch script:

```bash
export SDA_S3_BUCKET=amzn-s3-demo-bucket
scripts/.venv/bin/python qwen_image_edit/generate_synthetic.py \
    --placement hazardous \
    --dataset-prefix datasets/ablation_hazardous \
    --output-suffix hazardous
```